# Throw Away 99% of the Data, Keep the Same Model

**Topic:** Support vectors &nbsp;·&nbsp; **Difficulty:** Medium

The defining property of an SVM is that the solution depends only on the points
touching or violating the margin. Every other training point could be deleted and
the hyperplane would not move an inch.

Verify it. Fit an SVM, identify the support vectors, refit on those alone, and
check the two decision functions agree.

## The data (created in the next cell)

- `X` `(1200, 8)`, `y` `(1200,)` in `{-1, +1}`, mostly well separated.
- Use `SVC(kernel="linear", C=1.0)` from sklearn.

## Your task

With **sklearn and NumPy**:

1. Fit `SVC(kernel="linear", C=1.0)` on the full training set.
2. Extract the support vector indices (`model.support_`).
3. Refit an identical `SVC` on **only** the support vectors.
4. Compare the decision function of both models on `X`.
5. Set `answers` to a dict with `"n_support"` (int), `"frac_support"` (float),
   `"max_decision_diff"` (largest absolute difference between the two decision
   functions on `X`), and `"same_predictions"` (`1.0` if all signs agree).

## Evaluation

Counts must match exactly. Your `max_decision_diff` must match the reference
value to a relative tolerance of 1e-6, and the evaluation cell also requires
it to be under 1e-2 with `same_predictions` equal to 1.0.

In [ ]:
import numpy as np

rng = np.random.default_rng(1111)
n, d = 1200, 8
y = rng.choice([-1, 1], n).astype(float)
direction = np.array([1.2, -0.9, 0.7, 0.4, 0.0, 0.0, 0.0, 0.0])
X = y[:, None] * direction + rng.normal(0, 0.55, (n, d))
print(f"X {X.shape}, class balance: {(y > 0).mean():.3f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("n_support", [], 12.0, -2.00925617777399, 1e-06),
    ("frac_support", [], 0.01, -0.0016743801481449916, 1e-06),
    ("max_decision_diff", [], 0.001251869820155349, -0.00020961059749299575, 1e-06),
    ("same_predictions", [], 1.0, -0.16743801481449916, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "max_decision_diff" in answers:
    _agree = answers["max_decision_diff"] < 1e-2 and answers["same_predictions"] == 1.0
    print(f"  {'refit reproduces model':<24} {'ok' if _agree else 'wrong'}  "
          f"(max diff = {answers['max_decision_diff']:.2e})")
    _ok = _ok and _agree

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The dual solution has `alpha_i = 0` for every point strictly outside the margin,
so those points contribute nothing to `w = sum alpha_i y_i x_i`. Refitting on the
support set alone reproduces the identical hyperplane, which is the theoretical
reason SVMs generalize well (the solution depends on few points) and the
practical reason kernel SVMs stay tractable at prediction time.

```python
import numpy as np
from sklearn.svm import SVC

full = SVC(kernel="linear", C=1.0).fit(X, y)
sv = full.support_

sub = SVC(kernel="linear", C=1.0).fit(X[sv], y[sv])

d_full = full.decision_function(X)
d_sub = sub.decision_function(X)

answers = {
    "n_support": int(len(sv)),
    "frac_support": float(len(sv) / len(y)),
    "max_decision_diff": float(np.abs(d_full - d_sub).max()),
    "same_predictions": float(np.all(np.sign(d_full) == np.sign(d_sub))),
}
```